# Orchestration
orchestration.py is the main file for creating the agentic system. This file's process is:
1. Initialize all specialist agents via their factory functions (manual)
2. Initialize the orchestrator agent (automatic)
3. Build the explicit workflow graph via `WorkflowBuilder` (automatic)
4. Run the workflow

## AG2 → MAF Migration Summary
| AG2 Pattern | MAF Equivalent |
|---|---|
| `ConversableAgent` + `LLMConfig` | `Agent` via `FoundryChatClient` |
| `AutoPattern` + `initiate_group_chat` | `WorkflowBuilder` with explicit `router_executor` |
| `OnCondition` + `StringLLMCondition` | LLM-driven routing inside `router_executor` |
| `handoffs.add_llm_conditions()` | `WorkflowBuilder.add_edge()` — explicit, PROPAGATOR-ready |
| `UserProxyAgent` | Removed — entry via `workflow.run()` |
| `registerExecution(user_proxy)` | Removed — tools registered directly on each `Agent` |
| `LLMConfig.from_json` | `FoundryChatClient` reading from Azure credentials |

## Why WorkflowBuilder?
In AG2, the agent communication graph was **implicit** — it lived inside `AutoPattern` and the routing logic.
To build PROPAGATOR's instrumentation layer in AG2 would require reverse-engineering the graph by wrapping message-passing hooks.

In MAF, the `WorkflowBuilder` graph is **explicit Python** — you can inspect it, modify it, and `add_edge()` programmatically.
This is the architectural reason for the migration.

## Path Setup
Since this notebook lives in a `docs/` subfolder, we adjust `sys.path` to resolve imports from the project root.

In [ ]:
import sys
from pathlib import Path

def setup_path():
    ai_root = Path().resolve().parent
    if str(ai_root) not in sys.path:
        sys.path.insert(0, str(ai_root))

setup_path()

## Imports
MAF imports replace the AG2 `autogen` imports entirely.
Each specialist agent is now imported as a **factory function** (`build_*_agent`) rather than a class.
This is consistent with MAF's pattern — agents are constructed via factories, not subclassed from a framework base.

- `agent_framework` replaces `autogen`
- `FoundryChatClient` replaces `LLMConfig.from_json` — reads Azure credentials from the environment
- `WorkflowBuilder`, `executor`, `WorkflowContext` replace `AutoPattern` + `initiate_group_chat`
- `BaseAgent` is no longer imported — refactored to a thin MAF wrapper; not needed at the orchestration level

In [ ]:
import asyncio
import json
from typing_extensions import Never

from agent_framework import Agent, WorkflowBuilder, executor, WorkflowContext
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

from adviceAgent import build_advice_agent
from dbAgent import build_db_agent, DatabaseConnection
from diagnosisAgent import build_diagnosis_agent
from priceAgent import build_price_agent
from statsAgent import build_stat_agent
from utils import load_prompts, load_safeguards
from messageCleanser import OutputCleanser

prompts = load_prompts()
safeguards = load_safeguards()

## Shared Client
All agents share a single `FoundryChatClient` instance backed by Azure credentials.
Swapping the model or endpoint here propagates to every agent automatically.

**AG2 equivalent:** `LLMConfig.from_json(path="OAI_CONFIG_LIST.json")` assigned to `assistant.agent.llm_config`

In AG2, the config had to be manually assigned to each agent after construction via `initAssistant()`.
In MAF, the client is passed at construction time inside each factory — no post-hoc assignment needed.

In [ ]:
client = FoundryChatClient(
    credential=AzureCliCredential(),
    # endpoint and deployment read from environment:
    # AZURE_AI_FOUNDRY_ENDPOINT, AZURE_AI_FOUNDRY_DEPLOYMENT
)

## Initialize Specialist Agents
Each specialist agent is constructed via its `build_*_agent()` factory.

**AG2 equivalent:** `initAssistant(AgentClass, key, *args)` — which constructed the agent then manually assigned `llm_config`.

In MAF, factories accept `prompts[key]` directly and the client is already baked in.
Agents requiring extra constructor args (e.g. `DBAgent` needs `DatabaseConnection`) pass them to their factory.

**This is the only cell that requires manual changes when adding or removing agents.**
Everything downstream (the router, the workflow graph) adapts automatically via `AGENT_REGISTRY`.

In [ ]:
advice_agent    = build_advice_agent(prompts["advice"])
db_agent        = build_db_agent(prompts["db"], DatabaseConnection())
diagnosis_agent = build_diagnosis_agent(prompts["diagnosis"])
price_agent     = build_price_agent(prompts["price"])
stat_agent      = build_stat_agent(prompts["stats"])

# AGENT_REGISTRY: the explicit graph node map.
# PROPAGATOR can inspect, iterate, or add edges to this dict directly.
# AG2 equivalent: [a.agent for a in assistants] — implicit, not inspectable.
AGENT_REGISTRY: dict[str, Agent] = {
    "AdviceAgent":    advice_agent,
    "DBAgent":        db_agent,
    "DiagnosisAgent": diagnosis_agent,
    "PriceAgent":     price_agent,
    "StatAgent":      stat_agent,
}

## Initialize Orchestrator
The orchestrator is an LLM-based routing agent. Its job is to inspect the user message and return a JSON routing decision.

**AG2 equivalent:** `ConversableAgent` with `handoffs.add_llm_conditions(conditions)` where each condition was an
`OnCondition(target=AgentTarget(...), condition=StringLLMCondition(prompt=description))`.

In AG2, this wiring was **implicit** — the framework interpreted conditions internally and routed behind the scenes.
In MAF, the routing decision is made explicitly inside `router_executor` (next cell), where the orchestrator LLM
call is visible Python code. This is what makes the graph inspectable and PROPAGATOR-hookable.

In [ ]:
orchestrator_agent = client.as_agent(
    name="orchestrator",
    instructions=prompts["orchestrator"]["instructions"],
    default_options={"temperature": 0.0},
)

## Safeguard Helper
**AG2 equivalent:** `apply_safeguard_policy(agents=[...], policy="safeguards.json", safeguard_llm_config=LLM_CONFIG)`

In AG2, safeguard policy was applied as a decorator over agents at initialization time.
In MAF, safeguard checking is an explicit gate inside `router_executor` — called before the orchestrator LLM runs.
This makes the security boundary visible and testable without needing to unwrap framework internals.

In [ ]:
def _passes_safeguards(message: str) -> bool:
    """
    Inline safeguard check against loaded safeguards.json policy.
    Extend with an LLM-based policy call if your safeguards require semantic evaluation.
    """
    blocked_patterns = safeguards.get("blocked_patterns", [])
    message_lower = message.lower()
    return not any(pattern.lower() in message_lower for pattern in blocked_patterns)

## Router Executor (The Explicit Graph Node)
This is the heart of the MAF orchestration and the key architectural difference from AG2.

**AG2 equivalent:** `AutoPattern` + `initiate_group_chat` + `OnCondition` + `StringLLMCondition` + `handoffs`

In AG2, all of this was hidden inside the framework. The routing decision happened inside `AutoPattern.__call__()`,
and could not be inspected or intercepted without monkey-patching.

In MAF, the routing decision is **right here** — an explicit `await orchestrator_agent.run(...)` call that returns
a JSON routing object. The selected agent is dispatched via a direct `await selected_agent.run(task)` call.

**This is where PROPAGATOR hooks live.** To instrument the graph:
- Add `WorkflowBuilder.add_edge('router', 'propagator_observer')` in `build_workflow()`
- Or wrap the `await selected_agent.run(task)` call with a tracing context

No reverse-engineering required.

In [ ]:
@executor(id="router")
async def router_executor(
    task: str,
    ctx: WorkflowContext[Never, str],
) -> None:
    output_cleanser = OutputCleanser()
    max_iterations = 10

    for _ in range(max_iterations):

        # Safeguard gate
        if not _passes_safeguards(task):
            await ctx.yield_output(
                "Your request was flagged as potentially violating our safety regulations. "
                "Please try again with a different prompt."
            )
            return

        # Orchestrator selects which specialist to route to.
        # Returns JSON: {"agent": "<AgentName>", "refined_task": "<...>"}
        routing_response = await orchestrator_agent.run(
            f"Given the following user message, return a JSON object with:\n"
            f"  'agent': one of {list(AGENT_REGISTRY.keys())}\n"
            f"  'refined_task': the message rewritten for that specialist\n\n"
            f"Message: {task}"
        )

        try:
            routing = json.loads(routing_response.text)
            selected_name = routing["agent"]
            refined_task  = routing.get("refined_task", task)
        except (json.JSONDecodeError, KeyError):
            await ctx.yield_output("I'm sorry, there was an error in the response. Please try again.")
            return

        if selected_name not in AGENT_REGISTRY:
            await ctx.yield_output("I'm sorry, there was an error in the response. Please try again.")
            return

        # Explicit graph edge — PROPAGATOR hooks live here.
        selected_agent = AGENT_REGISTRY[selected_name]
        specialist_result = await selected_agent.run(refined_task)
        raw_reply = specialist_result.text or ""

        if not raw_reply:
            await ctx.yield_output("I'm sorry, there was an error in the response. Please try again.")
            return

        clean_reply = output_cleanser.cleanOutput(raw_reply)

        print(f"ROUTING → {selected_name}")
        print(f"REPLY: {raw_reply}")
        print(f"CLEAN REPLY: {clean_reply}")

        await ctx.yield_output(clean_reply)
        return  # single-turn; remove return for multi-turn loops

    await ctx.yield_output("I'm sorry, there was an error in the response. Please try again.")

## Workflow Graph
**AG2 equivalent:** `AutoPattern(agents=[...], initial_agent=orchestratorAgent, group_manager_args={...}, user_agent=user)`

In AG2, `AutoPattern` constructed an implicit graph from the agents list.
You could not call `.add_edge()`, inspect the graph structure, or hook into transitions without patching the framework.

In MAF, `WorkflowBuilder` is explicit Python. The graph is a first-class object.
`add_edge()` calls here are where PROPAGATOR instrumentation nodes get wired in.

**To add a PROPAGATOR observer:**
```python
builder.add_edge("router", "propagator_observer")
```

In [ ]:
def build_workflow():
    builder = WorkflowBuilder(start_executor=router_executor)
    # PROPAGATOR: add_edge() calls go here
    return builder.build()

workflow = build_workflow()

## Orchestrate
**AG2 equivalent:**
```python
def orchestrate(message: str):
    result, final_context, last_agent = initiate_group_chat(
        pattern=agent_pattern, messages=message, max_rounds=10
    )
    return result
```

In MAF, `workflow.run()` replaces `initiate_group_chat()`. It is async-native.
The synchronous `orchestrate()` wrapper is kept so `userMessageBuilder.py` can call it without changes.
When ready to go fully async end-to-end, replace `orchestrate()` calls with `await orchestrate_async()`
and drop the `asyncio.run()` wrapper.

`max_rounds=10` is no longer a parameter — iteration is controlled by the `max_iterations` guard
inside `router_executor` directly.

In [ ]:
async def orchestrate_async(full_message_with_context: str) -> str:
    result = await workflow.run(full_message_with_context)
    return result

def orchestrate(full_message_with_context: str) -> str:
    """Synchronous wrapper for userMessageBuilder.py compatibility."""
    return asyncio.run(orchestrate_async(full_message_with_context))

## Example
Example usage of the orchestration — mirrors the AG2 original.

`chat_context` is no longer passed directly into `orchestrate()` since context injection
is handled upstream in `userMessageBuilder.py` via the `--CONTEXT--` block prepended to the message.
The `build_instructions()` method in `baseAgent.py` handles per-request context enrichment if needed.

In [ ]:
message = "I am a 55 year old pregnant woman who smokes, can you give me some healthcare advice?"

# Run via orchestrate() (sync wrapper) or orchestrate_async() in an async context
result = orchestrate(message)
print(result)